<h1> Introduction </h1>

Dans ce module, nous avons travaillé avec un dataset contenant des informations sur les films et les séries disponibles sur Netflix. Au cours du précédent notebook sur Pandas, nous avons déjà manipulé et modélisé ce dataset pour en extraire des informations pertinentes. Nous avons nettoyé les données, géré les valeurs manquantes, et créé des colonnes supplémentaires pour faciliter notre analyse.

Nous allons ici créer des visualisations avec Plotly afin d'offrir une vue d'ensemble des données que nous avons pu mettre en évidence précédemment et de potentiellement tirer des conclusions sur des problématiques business.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"


netflix = pd.read_csv('netflix_cleaning.csv')
netflix.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,listed_in,description,duration (movies),seasons (TV Show),year added,month added,day of the week added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,Documentaries,"As her father nears the end of his life, filmm...",90.0,NaN,2021.0,9.0,5.0
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",NaN,2.0,2021.0,9.0,4.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,NaN,1.0,2021.0,9.0,4.0
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,2021-09-24,2021,TV-MA,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",NaN,1.0,2021.0,9.0,4.0
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,NaN,2.0,2021.0,9.0,4.0


In [2]:
# Compter le nombre de films et de séries par année de sortie

sorties_par_annees = netflix['release_year'].value_counts().reset_index()

fig= px.histogram(sorties_par_annees, x='release_year', y='count', title = 'Nombre de films et de séries par année de sortie')
fig.show()

In [3]:
# Créer un graphique linéaire pour visualiser le nombre de films et séries ajoutés chaque mois

sorties_par_mois = netflix['month added'].value_counts().reset_index()
sorties_par_mois

fig= px.line(sorties_par_mois.sort_values('month added'), x='month added', y='count', title = 'Nombre de films et séries par mois de sortie')
fig.show()

In [4]:
# Créer un graphique en aires pour visualiser le nombre de films et séries ajoutés par année et mois

netflix['year_month'] = pd.to_datetime(netflix['date_added']).dt.to_period('M').astype('str')
sorties_par_year_month = netflix['year_month'].value_counts().reset_index()
sorties_par_year_month = sorties_par_year_month.sort_values('year_month')

fig = px.area(sorties_par_year_month, x='year_month', y='count', title = 'Nombre de films et de séries par année et mois')
fig.update_layout(xaxis_title = 'Année et Mois', yaxis_title = 'Nombre de films/séries')
fig.show()

In [5]:
# Créer un box plot pour visualiser la distribution de la durée des films

films = netflix[netflix['type'] == 'Movie']

fig = px.box(films, y = 'duration (movies)',title = 'Distribution de la durée des films')
fig.show()

In [6]:
# Créer un pie chart pour visualiser la répartition des films et séries par genre

categories = pd.read_csv('categories_exploded.csv')

shows_par_genre = categories['listed_in_list'].value_counts().reset_index()
shows_par_genre.columns = ['genre', 'count']

fig = px.pie(shows_par_genre, names='genre', values='count', title = 'Répartition des films et séries par genre')
fig.show()

In [7]:
# Le pie chart n'est visuellement pas très clair, du fait du nombre important de valeurs
#Créer plutôt un treemap pour visualiser la répartition des genres des séries

fig = px.treemap(shows_par_genre, path = ['genre'],values='count', title = 'Répartition des films et séries par genre')
fig.show()

In [8]:
# Créer un scatter plot pour visualiser la répartition des séries par nombre de saisons et année de sortie

series = netflix[netflix['type']=='TV Show']

fig = px.scatter(series, x='release_year', y='seasons (TV Show)', size = 'seasons (TV Show)', 
                 title = 'Répartition des series par nombre de saisons et année de sortie',
                labels = {'release_year': 'année de sortie', 'seasons (TV Show)': 'nombre de saisons'})
fig.show()

In [9]:
# Créer une carte choroplèthe pour visualiser la répartition des films et séries par pays

#on prépare d'abord les données

pays = pd.read_csv('coutries_exploded.csv')
codes_iso = pd.read_csv('country_iso_codes.csv')

merged_df = pays.merge(codes_iso, left_on = 'countries', right_on = 'Country', how = 'left')

shows_par_pays_iso = merged_df.groupby(['countries', 'ISO Alpha-3 Code']).size().reset_index(name = 'show counts')
shows_par_pays_iso

,countries,ISO Alpha-3 Code,show counts
0,Afghanistan,AFG,1
1,Albania,ALB,1
2,Algeria,DZA,3
3,Angola,AGO,1
4,Argentina,ARG,91
...,...,...,...
108,Uruguay,URY,14
109,Vatican City,VAT,1
110,Venezuela,VEN,4
111,Vietnam,VNM,7


In [10]:
#puis on crée la carte choroplèthe

fig = px.choropleth(shows_par_pays_iso, locations = 'ISO Alpha-3 Code',
                    color = 'show counts',
                    title = 'Répartition des films et séries par pays',
                    labels = {'show counts': 'Nombre de films/séries'},
                    color_continuous_scale = 'aggrnyl'
                   )

# on modifie la figure en choisissant sa taille ainsi que son titre 
fig.update_layout(
    height=600, 
    width=1000,
    margin={"r":0,"t":50,"l":0,"b":0}, # t:50 laisse un peu de place pour le titre
    title_x=0.45 # Centre le titre
)


fig.show()